In [1]:
!pip install -q networkx
!pip install -q python-louvain
!pip install -q tqdm
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.5 MB/s eta 0:00:00


# Import Libraries

In [2]:
import os
import shutil
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

import networkx as nx
# import community as community_louvain
from community import community_louvain

In [ ]:
# DATASET_PATH = "/content"

# import zipfile
# import os

# zip_path = f"/content/kvasir-seg.zip"

# extract_path = DATASET_PATH

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)

# Download Dataset

In [3]:
import os
import zipfile

URL = "https://raw.githubusercontent.com/nazmul-1117/polyp-dataset-pruning/main/data/kvasir-seg.zip"

ZIP_PATH = "/content/kvasir-seg.zip"
DATASET_PATH = "/content/kvasir-seg"

# Download
!wget -q --show-progress "{URL}" -O "{ZIP_PATH}"

# Extract
os.makedirs(DATASET_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATASET_PATH)

print("Done!")

/content/kvasir-seg 100%[===================>]  44.03M  --.-KB/s    in 0.1s    
Done!


# Inp/Opt Directory

In [4]:
input_dir = "/content/kvasir-seg/Kvasir-SEG/images"
output_dir = "/content/pruned_images"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


# Transform Image

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
class PolypDataset(Dataset):

    def __init__(self, image_dir, transform=None):

        self.image_dir = image_dir
        self.transform = transform

        self.image_paths = sorted([
            os.path.join(image_dir,f)
            for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg",".jpeg",".png"))
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        path = self.image_paths[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, path

In [ ]:
dataset = PolypDataset(
    input_dir,
    transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(len(dataset))

1000


# Cell 8 — Load DINOv2

In [ ]:
model = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vitb14"
)

model.eval()
model.to(device)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 298MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((768,), eps=1e-06, elementwise_affi

# Cell 9 — Feature Extraction

In [ ]:
embeddings = []
image_paths = []

with torch.no_grad():

    for images, paths in tqdm(loader):

        images = images.to(device)

        features = model(images)

        embeddings.append(features.cpu())

        image_paths.extend(paths)

100%|██████████| 32/32 [00:16<00:00,  1.96it/s]


In [ ]:
embeddings = torch.cat(embeddings)
print(embeddings.shape)

torch.Size([1000, 768])


# Cell 10 — Normalize

In [ ]:
embeddings = F.normalize(
    embeddings,
    p=2,
    dim=1
)

# Cell 11 — Similarity Matrix

In [ ]:
similarity = embeddings @ embeddings.T

print(similarity.shape)

torch.Size([1000, 1000])


# Cell 12 — Threshold Graph

In [ ]:
tau = 0.80

G = nx.Graph()

N = len(image_paths)

G.add_nodes_from(range(N))

for i in range(N):

    for j in range(i+1,N):

        score = similarity[i,j].item()

        if score >= tau:

            G.add_edge(
                i,
                j,
                weight=score
            )

# Cell 13 — Louvain

In [ ]:
partition = community_louvain.best_partition(
    G,
    weight='weight'
)

In [ ]:
num_communities = len(set(partition.values()))

print(num_communities)

34


# Cell 14 — Group Nodes

In [ ]:
from collections import defaultdict

communities = defaultdict(list)

for node, cid in partition.items():

    communities[cid].append(node)

# Cell 15 — Representative (Medoid)

In [ ]:
representatives = []

for nodes in communities.values():

    if len(nodes) == 1:

        representatives.append(nodes[0])
        continue

    sub = similarity[nodes][:,nodes]

    mean_sim = sub.mean(dim=1)

    representative = nodes[mean_sim.argmax().item()]

    representatives.append(representative)

# Cell 16 — Save Images

In [ ]:
os.makedirs(output_dir, exist_ok=True)

for idx in representatives:

    shutil.copy(
        image_paths[idx],
        output_dir
    )

# Cell 17 — Report

In [ ]:
original = len(image_paths)

selected = len(representatives)

ratio = 100*(1-selected/original)

print("="*40)
print(f"Original Images : {original}")
print(f"Communities     : {len(communities)}")
print(f"Selected Images : {selected}")
print(f"Compression     : {ratio:.2f}%")
print("="*40)

Original Images : 1000
Communities     : 34
Selected Images : 34
Compression     : 96.60%


# Nex

In [ ]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from community import community_louvain  # python-louvain package
from torch.utils.data import Dataset, DataLoader, Subset
from skimage.metrics import structural_similarity as ssim

# =====================================================================
# MODULE 1: ARCHITECTURE DEFINITION (U-NET)
# =====================================================================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Downsampling Path
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Upsampling Path
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]

            if x.shape != skip_connection.shape:
                x = torch.nn.functional.interpolate(x, size=skip_connection.shape[2:])

            concat_x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_x)

        return self.final_conv(x)


# =====================================================================
# MODULE 2: PIPELINE ENGINE (PRUNER & EXPERIMENT MANAGER)
# =====================================================================

class DatasetPruner:
    """Manages training-free graph dataset pruning to prevent data leakage."""
    def __init__(self, dataset: Dataset, train_indices: list, tau: float = 0.85, p: float = 0.10):
        self.dataset = dataset
        self.train_indices = train_indices
        self.tau = tau
        self.p = p

    def _compute_ssim_matrix(self) -> np.ndarray:
        n_train = len(self.train_indices)
        ssim_matrix = np.eye(n_train)

        # Cache pulled images converted to grayscale arrays
        cached_images = []
        for idx in self.train_indices:
            img, _ = self.dataset[idx]
            if isinstance(img, torch.Tensor):
                img = img.permute(1, 2, 0).cpu().numpy()
            img_gray = np.mean(img, axis=2) if img.ndim == 3 else img
            # Ensure static bounds for metric stability
            img_normalized = (img_gray - img_gray.min()) / (img_gray.max() - img_gray.min() + 1e-5)
            cached_images.append(img_normalized)

        for i in range(n_train):
            for j in range(i + 1, n_train):
                score = ssim(cached_images[i], cached_images[j], data_range=1.0)
                ssim_matrix[i, j] = score
                ssim_matrix[j, i] = score
        return ssim_matrix

    def execute_pruning(self) -> list:
        print("[Engine] Initializing Structural Manifold Characterization...")
        ssim_matrix = self._compute_ssim_matrix()

        # Build network topology tracking explicitly on subset coordinates
        adj_matrix = (ssim_matrix >= self.tau).astype(int)
        G = nx.from_numpy_array(adj_matrix)

        print("[Engine] Maximizing Network Modularity Partitioning...")
        partition = community_louvain.best_partition(G)

        communities = {}
        for node, comm_id in partition.items():
            communities.setdefault(comm_id, []).append(node)

        retained_subset_local = []
        for comm_id, nodes in communities.items():
            if len(nodes) == 1:
                retained_subset_local.append(nodes[0])
            else:
                # Rank nodes by local subgraph degree centrality within the cluster
                degrees = dict(G.degree(nodes))
                sorted_nodes = sorted(nodes, key=lambda n: degrees[n], reverse=True)
                selection_budget = max(1, int(np.ceil(self.p * len(nodes))))
                retained_subset_local.extend(sorted_nodes[:selection_budget])

        # Map local configuration indices securely back to main dataset indices
        pruned_train_indices = [self.train_indices[idx] for idx in retained_subset_local]
        print(f"[Engine] Manifold Optimization Complete: Reduced {len(self.train_indices)} to {len(pruned_train_indices)} sample paths.")
        return pruned_train_indices


class TrainingEngine:
    """Class-based pipeline execution framework tracking performance metrics."""
    def __init__(self, model: nn.Module, device: torch.device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-4)

    def compute_metrics(self, pred: torch.Tensor, target: torch.Tensor):
        # Convert raw logits to binary segmentation maps
        probs = torch.sigmoid(pred)
        preds = (probs > 0.5).float()

        intersection = (preds * target).sum(dim=(2, 3))
        total = preds.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        union = total - intersection

        dice = (2.0 * intersection / (total + 1e-5)).mean().item()
        iou = ((intersection + 1e-5) / (union + 1e-5)).mean().item()
        return dice, iou

    def train_epoch(self, loader: DataLoader) -> float:
        self.model.train()
        total_loss = 0.0
        for imgs, masks in loader:
            imgs, masks = imgs.to(self.device), masks.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(imgs)
            loss = self.criterion(outputs, masks)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * imgs.size(0)
        return total_loss / len(loader.dataset)

    def evaluate(self, loader: DataLoader):
        self.model.eval()
        running_dice, running_iou = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in loader:
                imgs, masks = imgs.to(self.device), masks.to(self.device)
                outputs = self.model(imgs)
                d_score, i_score = self.compute_metrics(outputs, masks)
                running_dice += d_score * imgs.size(0)
                running_iou += i_score * imgs.size(0)
        return running_dice / len(loader.dataset), running_iou / len(loader.dataset)


# =====================================================================
# MODULE 3: SYNTHETIC DATA & EXPERIMENTAL EXECUTION (MOCK RIG)
# =====================================================================

class MockKvasirDataset(Dataset):
    """Synthetic generator emulating structural properties of Kvasir-SEG."""
    def __init__(self, num_samples=100, channels=3, height=128, width=128):
        self.num_samples = num_samples
        # Inject systematic random structure to mimic structural redundancy
        self.data = torch.randn(num_samples, channels, height, width)
        self.masks = torch.randint(0, 2, (num_samples, 1, height, width)).float()

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return self.data[idx], self.masks[idx]

if __name__ == "__main__":
    # Fix execution environment seed variables
    torch.manual_seed(42)
    np.random.seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("[Pipeline] Loading Target Mandate Dataset Environment...")
    full_dataset = MockKvasirDataset(num_samples=120)

    # Enforce clear separation of partitions prior to processing
    all_indices = list(range(120))
    train_split, test_split = all_indices[:90], all_indices[90:]
    test_loader = DataLoader(Subset(full_dataset, test_split), batch_size=8, shuffle=False)

    # -----------------------------------------------------------------
    # Configuration A: Baseline Run (Full Training Dataset Execution)
    # -----------------------------------------------------------------
    print("\n--- Running Baseline Configuration (Full Training Set) ---")
    full_train_loader = DataLoader(Subset(full_dataset, train_split), batch_size=8, shuffle=True)
    baseline_model = UNet()
    engine_full = TrainingEngine(baseline_model, device)

    t_start_full = time.time()
    for epoch in range(1, 3):  # Scale epoch validation as required
        loss = engine_full.train_epoch(full_train_loader)
    t_end_full = time.time()
    t_full_total = t_end_full - t_start_full

    f_dice, f_iou = engine_full.evaluate(test_loader)
    print(f"Full Dataset Train Time: {t_full_total:.2f}s | Test Dice: {f_dice:.4f} | Test IoU: {f_iou:.4f}")

    # -----------------------------------------------------------------
    # Configuration B: Optimized Run (PRIME Pipeline Dataset Pruning)
    # -----------------------------------------------------------------
    print("\n--- Running Optimized Configuration (PRIME Topology Pipeline) ---")
    pruner = DatasetPruner(full_dataset, train_split, tau=0.10, p=0.20)
    pruned_indices = pruner.execute_pruning()

    pruned_train_loader = DataLoader(Subset(full_dataset, pruned_indices), batch_size=8, shuffle=True)
    pruned_model = UNet()
    engine_pruned = TrainingEngine(pruned_model, device)

    t_start_pruned = time.time()
    for epoch in range(1, 3):
        loss = engine_pruned.train_epoch(pruned_train_loader)
    t_end_pruned = time.time()
    t_pruned_total = t_end_pruned - t_start_pruned

    p_dice, p_iou = engine_pruned.evaluate(test_loader)
    print(f"Pruned Dataset Train Time: {t_pruned_total:.2f}s | Test Dice: {p_dice:.4f} | Test IoU: {p_iou:.4f}")

    # -----------------------------------------------------------------
    # Efficiency Assessment
    # -----------------------------------------------------------------
    print("\n================ FINAL REPORT SUMMARY ================")
    print(f"Baseline Full Training Time (T_full): {t_full_total:.3f} seconds.")
    print(f"Pruned Pipeline Training Time (T_pruned): {t_pruned_total:.3f} seconds.")
    print(f"Calculated Efficiency Coefficient delta_T: {(((t_full_total - t_pruned_total)/t_full_total)*100):.2f}% Reduction.")

[Pipeline] Loading Target Mandate Dataset Environment...

--- Running Baseline Configuration (Full Training Set) ---
Full Dataset Train Time: 3.74s | Test Dice: 0.5610 | Test IoU: 0.3899

--- Running Optimized Configuration (PRIME Topology Pipeline) ---
[Engine] Initializing Structural Manifold Characterization...
[Engine] Maximizing Network Modularity Partitioning...
[Engine] Manifold Optimization Complete: Reduced 90 to 90 sample paths.
Pruned Dataset Train Time: 3.11s | Test Dice: 0.5636 | Test IoU: 0.3924

================ FINAL REPORT SUMMARY ================
Baseline Full Training Time (T_full): 3.735 seconds.
Pruned Pipeline Training Time (T_pruned): 3.107 seconds.
Calculated Efficiency Coefficient delta_T: 16.82% Reduction.


# NewY

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import transforms

class DinoEmbeddingGraphEngine:
    """Extracts DINOv2 features and constructs semantic graph topologies."""
    def __init__(self, device: torch.device):
        self.device = device
        print("[DINOv2 Engine] Initializing Frozen ViT Backbone (dinov2_vits14)...")
        # Using the small variant (ViT-S/14) which outputs a clean 384 or 768-dim embedding depending on setup
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.model.eval()

        # DINOv2 expects ImageNet normalization profiles
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def compute_semantic_similarity_matrix(self, dataset, train_indices: list) -> np.ndarray:
        embeddings = []

        print("[DINOv2 Engine] Generating Latent Feature Embeddings...")
        for idx in train_indices:
            img, _ = dataset[idx]
            # Assumes img is a PyTorch Tensor [3, H, W] scaled between [0, 1]
            if img.shape[0] == 1: # Convert grayscale instances to 3-channel if any exist
                img = img.repeat(3, 1, 1)

            input_tensor = self.transform(img).unsqueeze(0).to(self.device)

            # Extract the global [CLS] token representation
            cls_embedding = self.model(input_tensor) # Shape: [1, 384] or [1, 768]

            # Normalize embedding vector directly to streamline cosine calculation
            cls_embedding = nn.functional.normalize(cls_embedding, p=2, dim=1)
            embeddings.append(cls_embedding.cpu().numpy().flatten())

        embeddings = np.array(embeddings) # Shape: [N_train, Dimensions]

        print("[DINOv2 Engine] Computing Pairwise Cosine Commutations...")
        # Because vectors are pre-normalized, dot product is identical to Cosine Similarity
        cosine_matrix = np.dot(embeddings, embeddings.T)

        return cosine_matrix

# NXYZ

In [5]:
import os
import zipfile
import time
import glob
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from PIL import Image
from community import community_louvain
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

# =====================================================================
# MODULE 1: DATASET PREPARATION & HANDLING
# =====================================================================

class KvasirSegDataset(Dataset):
    """Custom Dataset wrapper for parsing unzipped Kvasir-SEG contents."""
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)
        self.transform = transform

        # Internal transformation to force geometric uniformity for tensors
        self.base_tensor_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Grayscale for ground truth

        if self.transform:
            # Apply identical random states to both images and targets if augmenting
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed)
            image_t = self.transform(image)
            torch.manual_seed(seed)
            mask_t = self.transform(mask)
        else:
            image_t = self.base_tensor_transform(image)
            mask_t = self.base_tensor_transform(mask)

        # Binarize the mask explicitly to 0.0 or 1.0
        mask_t = (mask_t > 0.5).float()
        return image_t, mask_t


def prepare_kvasir_data(zip_path, extract_to='./kvasir_data'):
    """Extracts zip directory structures securely into distinct clean paths."""
    if not os.path.exists(extract_to):
        print(f"[Data] Extracting dataset archive: {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

    # Locate paths dynamically based on common Kaggle directory naming structures
    search_img = os.path.join(extract_to, "**", "images", "*.jpg")
    search_masks = os.path.join(extract_to, "**", "masks", "*.jpg")

    # Fallback to PNG extensions if present
    img_paths = glob.glob(search_img, recursive=True) or glob.glob(os.path.join(extract_to, "**", "images", "*.png"), recursive=True)
    mask_paths = glob.glob(search_masks, recursive=True) or glob.glob(os.path.join(extract_to, "**", "masks", "*.png"), recursive=True)

    print(f"[Data] Found {len(img_paths)} images and {len(mask_paths)} segmentation masks.")
    return img_paths, mask_paths


# =====================================================================
# MODULE 2: DINOv2 MANIFOLD EXTRACTION & PRUNING ENGINE
# =====================================================================

class DinoEmbeddingGraphPruner:
    """Extracts DINOv2 visual features and runs Louvain community selection."""
    def __init__(self, dataset: Dataset, train_indices: list, device: torch.device):
        self.dataset = dataset
        self.train_indices = train_indices
        self.device = device

        print("[DINOv2 Engine] Initializing Frozen ViT Backbone (dinov2_vits14)...")
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
        self.model.eval()

        # Standard ImageNet scaling parameters required by DINOv2
        self.dino_transform = transforms.Compose([
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def prune(self, tau: float = 0.65, p: float = 0.20) -> list:
        embeddings = []
        print("[DINOv2 Engine] Computing Semantic Latent Spaces for Training Partition...")

        for idx in self.train_indices:
            img, _ = self.dataset[idx]
            # Input shape: [3, 224, 224] -> normalized and batched
            input_tensor = self.dino_transform(img).unsqueeze(0).to(self.device)

            # Extract Global CLS Token Vector
            cls_vec = self.model(input_tensor)
            cls_vec = nn.functional.normalize(cls_vec, p=2, dim=1)
            embeddings.append(cls_vec.cpu().numpy().flatten())

        embeddings = np.array(embeddings) # Matrix of shape [N_train, 384]

        # Generate the continuous manifold using Cosine Similarity Dot Products
        cosine_sim_matrix = np.dot(embeddings, embeddings.T)

        # Map values from [-1, 1] securely to an adjacency range of [0, 1]
        adj_matrix = (cosine_sim_matrix + 1.0) / 2.0

        # Structural sparsification via threshold mapping
        binary_adj = (adj_matrix >= tau).astype(int)

        # Build topological graph space
        G = nx.from_numpy_array(binary_adj)

        print("[DINOv2 Engine] Solving Network Modularity via Louvain Partitions...")
        partition = community_louvain.best_partition(G)

        communities = {}
        for node, comm_id in partition.items():
            communities.setdefault(comm_id, []).append(node)

        selected_local_nodes = []
        for comm_id, nodes in communities.items():
            if len(nodes) <= 1:
                selected_local_nodes.append(nodes[0])
            else:
                # Rank local cluster frames based on their node degree centrality
                degrees = dict(G.degree(nodes))
                sorted_nodes = sorted(nodes, key=lambda n: degrees[n], reverse=True)

                budget = max(1, int(np.ceil(p * len(nodes))))
                selected_local_nodes.extend(sorted_nodes[:budget])

        # Remap localized positions back to global dataset index coordinates
        pruned_global_indices = [self.train_indices[i] for i in selected_local_nodes]
        print(f"[DINOv2 Engine] Pruning complete. Budget reduced from {len(self.train_indices)} to {len(pruned_global_indices)} samples.")
        return pruned_global_indices


# =====================================================================
# MODULE 3: SEGMENTATION ARCHITECTURE & PIPELINE TRAINING
# =====================================================================

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class CompactUNet(nn.Module):
    """Standard U-Net encoder-decoder network optimized for training validation."""
    def __init__(self, in_c=3, out_c=1, features=[64, 128, 256]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        for feat in features:
            self.downs.append(DoubleConv(in_c, feat))
            in_c = feat

        for feat in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feat*2, feat, 2, stride=2))
            self.ups.append(DoubleConv(feat*2, feat))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final = nn.Conv2d(features[0], out_c, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip = skips[idx//2]
            if x.shape != skip.shape:
                x = torch.nn.functional.interpolate(x, size=skip.shape[2:])
            x = self.ups[idx+1](torch.cat((skip, x), dim=1))
        return self.final(x)


class ExecutionEngine:
    """Handles standard PyTorch loops, metrics calculation, and evaluation."""
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-3)

    def stats(self, pred, target):
        preds = (torch.sigmoid(pred) > 0.5).float()
        inter = (preds * target).sum(dim=(2, 3))
        total = preds.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = ((2.0 * inter) / (total + 1e-5)).mean().item()
        iou = ((inter + 1e-5) / (total - inter + 1e-5)).mean().item()
        return dice, iou

    def fit(self, loader):
        self.model.train()
        for imgs, masks in loader:
            imgs, masks = imgs.to(self.device), masks.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(imgs), masks)
            loss.backward()
            self.optimizer.step()

    def evaluate(self, loader):
        self.model.eval()
        d, i = 0.0, 0.0
        with torch.no_grad():
            for imgs, masks in loader:
                imgs, masks = imgs.to(self.device), masks.to(self.device)
                md, mi = self.stats(self.model(imgs), masks)
                d += md * imgs.size(0)
                i += mi * imgs.size(0)
        return d / len(loader.dataset), i / len(loader.dataset)


# =====================================================================
# MAIN EXPERIMENT EXECUTION PIPELINE
# =====================================================================

if __name__ == "__main__":
    # 0. System Initialization
    torch.manual_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[System] Executing computational runtime on: {device}")

    # 1. Dataset Parsing Configuration
    # ADJUST THIS PATH TO MATCH YOUR UPLOADED ZIP FILE NAME
    # ZIP_ARCHIVE_PATH = "Kvasir-SEG.zip"
    ZIP_ARCHIVE_PATH = "/content/kvasir-seg.zip"

    if not os.path.exists(ZIP_ARCHIVE_PATH):
        raise FileNotFoundError(f"Please upload your Kaggle zip file to Colab and update ZIP_ARCHIVE_PATH to point to it.")

    img_paths, mask_paths = prepare_kvasir_data(ZIP_ARCHIVE_PATH)
    base_dataset = KvasirSegDataset(img_paths, mask_paths)

    # 2. Strict Train/Test Separation Strategy (80/20)
    num_samples = len(base_dataset)
    indices = np.random.permutation(num_samples)
    split_boundary = int(num_samples * 0.8)

    train_indices = list(indices[:split_boundary])
    test_indices = list(indices[split_boundary:])

    test_loader = DataLoader(Subset(base_dataset, test_indices), batch_size=16, shuffle=False)

    # 3. Generate Graph-Pruned Dataset Partition
    pruner = DinoEmbeddingGraphPruner(base_dataset, train_indices, device)
    # tau: Similarity filter cutoff. p: top-centrality retention budget per community.
    pruned_train_indices = pruner.prune(tau=0.75, p=0.30)

    # 4. Benchmarking Environment Execution
    # Loader A: Full Baseline Training System
    full_train_loader = DataLoader(Subset(base_dataset, train_indices), batch_size=16, shuffle=True)
    # Loader B: Pruned Optimized System
    pruned_train_loader = DataLoader(Subset(base_dataset, pruned_train_indices), batch_size=16, shuffle=True)

    EPOCHS = 10  # Scale this as needed for your final thesis experiments

    # --- EXPERIMENT 1: BASELINE WORKFLOW ---
    print("\n[Pipeline] Training Model on FULL Kvasir Dataset...")
    full_model = CompactUNet()
    engine_full = ExecutionEngine(full_model, device)

    t0 = time.time()
    for epoch in range(EPOCHS):
        engine_full.fit(full_train_loader)
    t_full = time.time() - t0
    dice_full, iou_full = engine_full.evaluate(test_loader)

    # --- EXPERIMENT 2: PRUNED WORKFLOW ---
    print("\n[Pipeline] Training Model on PRUNED Kvasir Dataset...")
    pruned_model = CompactUNet()
    engine_pruned = ExecutionEngine(pruned_model, device)

    t1 = time.time()
    for epoch in range(EPOCHS):
        engine_pruned.fit(pruned_train_loader)
    t_pruned = time.time() - t1
    dice_pruned, iou_pruned = engine_pruned.evaluate(test_loader)

    # =====================================================================
    # FINAL STATISTICAL THESIS REPORTING
    # =====================================================================
    print("\n" + "="*50)
    print("               FINAL EXPERIMENTAL REPORT")
    print("="*50)
    print(f"Original Training Set Count : {len(train_indices)}")
    print(f"Pruned Training Set Count   : {len(pruned_train_indices)}")
    print(f"Dataset Size Reduction Ratio: {((1.0 - len(pruned_train_indices)/len(train_indices))*100):.2f}%")
    print("-"*50)
    print(f"Full Dataset Train Time     : {t_full:.2f} seconds")
    print(f"Pruned Dataset Train Time   : {t_pruned:.2f} seconds")
    print(f"Training Acceleration delta : {(((t_full - t_pruned) / t_full) * 100):.2f}% Faster")
    print("-"*50)
    print(f"Full Dataset Metric Score   : Mean Dice: {dice_full:.4f} | Mean IoU: {iou_full:.4f}")
    print(f"Pruned Dataset Metric Score : Mean Dice: {dice_pruned:.4f} | Mean IoU: {iou_pruned:.4f}")
    print(f"Absolute Dice Variance Delta: {(dice_full - dice_pruned):.4f}")
    print("="*50)

[System] Executing computational runtime on: cuda
[Data] Extracting dataset archive: /content/kvasir-seg.zip...
[Data] Found 1000 images and 1000 segmentation masks.
[DINOv2 Engine] Initializing Frozen ViT Backbone (dinov2_vits14)...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 230MB/s]


[DINOv2 Engine] Computing Semantic Latent Spaces for Training Partition...
[DINOv2 Engine] Solving Network Modularity via Louvain Partitions...
[DINOv2 Engine] Pruning complete. Budget reduced from 800 to 242 samples.

[Pipeline] Training Model on FULL Kvasir Dataset...

[Pipeline] Training Model on PRUNED Kvasir Dataset...

               FINAL EXPERIMENTAL REPORT
Original Training Set Count : 800
Pruned Training Set Count   : 242
Dataset Size Reduction Ratio: 69.75%
--------------------------------------------------
Full Dataset Train Time     : 282.11 seconds
Pruned Dataset Train Time   : 91.51 seconds
Training Acceleration delta : 67.56% Faster
--------------------------------------------------
Full Dataset Metric Score   : Mean Dice: 0.4249 | Mean IoU: 0.3043
Pruned Dataset Metric Score : Mean Dice: 0.0030 | Mean IoU: 0.0016
Absolute Dice Variance Delta: 0.4219
